In [3]:
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
df = pd.read_csv(r"C:\Crime_Incidents_in_2024.csv")  # Dosyanın tam yolunu kullan
df.head()

,X,Y,CCN,REPORT_DAT,SHIFT,METHOD,OFFENSE,BLOCK,XBLOCK,YBLOCK,...,BLOCK_GROUP,CENSUS_TRACT,VOTING_PRECINCT,LATITUDE,LONGITUDE,BID,START_DATE,END_DATE,OBJECTID,OCTO_RECORD_ID
0,399622.2700,134352.62,24093246,2024/06/19 14:52:56+00,DAY,OTHERS,THEFT/OTHER,1100 - 1199 BLOCK OF NEW JERSEY AVENUE SE,399622.270000,134352.620000,...,007203 1,7203.0,Precinct 131,38.877004,-77.004353,CAPITOL RIVERFRONT,2024/06/19 14:50:00+00,2024/06/19 14:52:00+00,679448603,NaN
1,398579.1351,134828.40,24131497,2024/08/27 01:13:37+00,EVENING,OTHERS,THEFT F/AUTO,300 - 399 BLOCK OF G STREET SW,398579.135069,134828.399995,...,010500 2,10500.0,Precinct 128,38.881289,-77.016376,SOUTHWEST,2024/08/26 18:51:00+00,2024/08/26 20:14:00+00,679448604,NaN
2,396836.0500,139850.46,24120734,2024/08/07 07:10:35+00,MIDNIGHT,OTHERS,THEFT/OTHER,16TH STREET NW AND COLUMBIA ROAD NW,396836.050011,139850.459999,...,003901 1,3901.0,Precinct 39,38.926525,-77.036488,NaN,2024/08/04 01:00:00+00,2024/08/04 01:30:00+00,679449098,NaN
3,398010.0800,138818.94,24168124,2024/10/30 03:40:30+00,MIDNIGHT,OTHERS,THEFT/OTHER,2000 - 2099 BLOCK OF 8TH STREET NW,398010.080000,138818.940000,...,003500 3,3500.0,Precinct 37,38.917236,-77.022946,NaN,2024/10/29 04:00:00+00,2024/10/29 20:00:00+00,679449103,NaN
4,397424.6800,141258.25,24172277,2024/11/05 23:15:06+00,EVENING,OTHERS,MOTOR VEHICLE THEFT,3900 - 3999 BLOCK OF 13TH STREET NW,397424.680000,141258.250000,...,002503 1,2503.0,Precinct 47,38.939209,-77.029705,NaN,2024/11/05 22:22:00+00,NaN,679449104,NaN


In [21]:
#
drop_columns = ["X", "Y", "XBLOCK", "YBLOCK", "OBJECTID", 
                "BLOCK_GROUP", "CENSUS_TRACT", "VOTING_PRECINCT"]

# Sadece bulunan sütunları sil
df_cleaned = df.drop(columns=[col for col in drop_columns if col in df.columns])


df_cleaned.head()


,WARD,ANC,DISTRICT,PSA,NEIGHBORHOOD_CLUSTER,LATITUDE,LONGITUDE,BID,YEAR,MONTH,...,METHOD_KNIFE,METHOD_OTHERS,OFFENSE_ASSAULT W/DANGEROUS WEAPON,OFFENSE_BURGLARY,OFFENSE_HOMICIDE,OFFENSE_MOTOR VEHICLE THEFT,OFFENSE_ROBBERY,OFFENSE_SEX ABUSE,OFFENSE_THEFT F/AUTO,OFFENSE_THEFT/OTHER
0,8.0,8F,1.0,106.0,Cluster 27,38.877004,-77.004353,CAPITOL RIVERFRONT,2024.0,6.0,...,False,True,False,False,False,False,False,False,False,True
1,6.0,6D,1.0,103.0,Cluster 9,38.881289,-77.016376,SOUTHWEST,2024.0,8.0,...,False,True,False,False,False,False,False,False,True,False
2,1.0,1C,3.0,302.0,Cluster 2,38.926525,-77.036488,NaN,2024.0,8.0,...,False,True,False,False,False,False,False,False,False,True
3,1.0,1B,3.0,305.0,Cluster 3,38.917236,-77.022946,NaN,2024.0,10.0,...,False,True,False,False,False,False,False,False,False,True
4,4.0,4C,NaN,404.0,Cluster 18,38.939209,-77.029705,NaN,2024.0,11.0,...,False,True,False,False,False,True,False,False,False,False


In [23]:
print(df_cleaned.isnull().sum())


WARD                                      4
ANC                                       4
DISTRICT                                745
PSA                                     468
NEIGHBORHOOD_CLUSTER                      4
LATITUDE                                  0
LONGITUDE                                 0
BID                                   23881
YEAR                                      6
MONTH                                     6
DAY                                       6
HOUR                                      6
SHIFT_EVENING                             0
SHIFT_MIDNIGHT                            0
METHOD_KNIFE                              0
METHOD_OTHERS                             0
OFFENSE_ASSAULT W/DANGEROUS WEAPON        0
OFFENSE_BURGLARY                          0
OFFENSE_HOMICIDE                          0
OFFENSE_MOTOR VEHICLE THEFT               0
OFFENSE_ROBBERY                           0
OFFENSE_SEX ABUSE                         0
OFFENSE_THEFT F/AUTO            

In [31]:
df_cleaned['WARD'] = df_cleaned['WARD'].fillna(df_cleaned['WARD'].mode()[0])
df_cleaned['ANC'] = df_cleaned['ANC'].fillna(df_cleaned['ANC'].mode()[0])
df_cleaned['NEIGHBORHOOD_CLUSTER'] = df_cleaned['NEIGHBORHOOD_CLUSTER'].fillna(df_cleaned['NEIGHBORHOOD_CLUSTER'].mode()[0])
df_cleaned['YEAR'] = df_cleaned['YEAR'].fillna(df_cleaned['YEAR'].median())
df_cleaned['MONTH'] = df_cleaned['MONTH'].fillna(df_cleaned['MONTH'].median())
df_cleaned['DAY'] = df_cleaned['DAY'].fillna(df_cleaned['DAY'].median())
df_cleaned['HOUR'] = df_cleaned['HOUR'].fillna(df_cleaned['HOUR'].median())



In [35]:
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()

# Örnek olarak 'WARD' ve 'NEIGHBORHOOD_CLUSTER' gibi kategorik sütunları encode ediyoruz
df_cleaned['WARD'] = label_encoder.fit_transform(df_cleaned['WARD'])
df_cleaned['NEIGHBORHOOD_CLUSTER'] = label_encoder.fit_transform(df_cleaned['NEIGHBORHOOD_CLUSTER'])



In [37]:
from sklearn.model_selection import train_test_split


X = df_cleaned.drop(columns=['OFFENSE_THEFT F/AUTO'])  
y = df_cleaned['OFFENSE_THEFT F/AUTO']  


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [41]:

for column in df_cleaned.columns:
    if df_cleaned[column].dtype == 'object':
        print(f"Sütun: {column} - Örnek Değerler: {df_cleaned[column].unique()}")


Sütun: ANC - Örnek Değerler: ['8F' '6D' '1C' '1B' '4C' '3F' '3A' '4B' '8A' '6B' '1A' '3C' '4A' '7F'
 '5C' '8D' '2A' '7D' '8E' '4E' '8C' '1E' '5F' '7E' '2F' '5D' '3B' '5B'
 '8B' '2C' '2B' '1D' '6E' '2D' '5E' '6C' '7B' '2G' '3E' '7C' '2E' '3/4G'
 '6A' '3D' '5A' '4D']


In [43]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()


df_cleaned['ANC'] = label_encoder.fit_transform(df_cleaned['ANC'])


In [47]:
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()
df_cleaned['ANC'] = label_encoder.fit_transform(df_cleaned['ANC'])

# Yeniden verilerinizi kontrol edin
print(df_cleaned['ANC'].head())


0    45
1    33
2     2
3     1
4    21
Name: ANC, dtype: int64


In [51]:
print(df_cleaned.columns)


Index(['WARD', 'ANC', 'DISTRICT', 'PSA', 'NEIGHBORHOOD_CLUSTER', 'LATITUDE',
       'LONGITUDE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'SHIFT_EVENING',
       'SHIFT_MIDNIGHT', 'METHOD_KNIFE', 'METHOD_OTHERS',
       'OFFENSE_ASSAULT W/DANGEROUS WEAPON', 'OFFENSE_BURGLARY',
       'OFFENSE_HOMICIDE', 'OFFENSE_MOTOR VEHICLE THEFT', 'OFFENSE_ROBBERY',
       'OFFENSE_SEX ABUSE', 'OFFENSE_THEFT F/AUTO', 'OFFENSE_THEFT/OTHER'],
      dtype='object')


In [57]:

print(y_train.value_counts())


ASSAULT
1    23435
Name: count, dtype: int64


In [63]:
print(df_crime.columns)


Index(['WARD', 'ANC', 'DISTRICT', 'PSA', 'NEIGHBORHOOD_CLUSTER', 'LATITUDE',
       'LONGITUDE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'SHIFT_EVENING',
       'SHIFT_MIDNIGHT', 'METHOD_KNIFE', 'METHOD_OTHERS',
       'OFFENSE_ASSAULT W/DANGEROUS WEAPON', 'OFFENSE_BURGLARY',
       'OFFENSE_HOMICIDE', 'OFFENSE_MOTOR VEHICLE THEFT', 'OFFENSE_ROBBERY',
       'OFFENSE_SEX ABUSE', 'OFFENSE_THEFT F/AUTO', 'OFFENSE_THEFT/OTHER',
       'ASSAULT'],
      dtype='object')


In [69]:

print(y_train.value_counts())


ASSAULT
1    23435
Name: count, dtype: int64


In [71]:

print(df_cleaned.columns)


Index(['WARD', 'ANC', 'DISTRICT', 'PSA', 'NEIGHBORHOOD_CLUSTER', 'LATITUDE',
       'LONGITUDE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'SHIFT_EVENING',
       'SHIFT_MIDNIGHT', 'METHOD_KNIFE', 'METHOD_OTHERS',
       'OFFENSE_ASSAULT W/DANGEROUS WEAPON', 'OFFENSE_BURGLARY',
       'OFFENSE_HOMICIDE', 'OFFENSE_MOTOR VEHICLE THEFT', 'OFFENSE_ROBBERY',
       'OFFENSE_SEX ABUSE', 'OFFENSE_THEFT F/AUTO', 'OFFENSE_THEFT/OTHER',
       'ASSAULT'],
      dtype='object')


In [73]:

crime_columns = [
    'OFFENSE_ASSAULT W/DANGEROUS WEAPON', 
    'OFFENSE_BURGLARY', 
    'OFFENSE_HOMICIDE', 
    'OFFENSE_MOTOR VEHICLE THEFT', 
    'OFFENSE_ROBBERY', 
    'OFFENSE_SEX ABUSE', 
    'OFFENSE_THEFT F/AUTO', 
    'OFFENSE_THEFT/OTHER', 
    'ASSAULT'
]


df_cleaned['CrimeType'] = df_cleaned[crime_columns].idxmax(axis=1)

# 'CrimeType' sütunundaki değerlerin dağılımına bakalım
print(df_cleaned['CrimeType'].value_counts())


CrimeType
OFFENSE_THEFT/OTHER                   13015
OFFENSE_THEFT F/AUTO                   6680
OFFENSE_MOTOR VEHICLE THEFT            5127
OFFENSE_ROBBERY                        2109
OFFENSE_ASSAULT W/DANGEROUS WEAPON     1026
OFFENSE_BURGLARY                       1004
OFFENSE_HOMICIDE                        187
OFFENSE_SEX ABUSE                       142
ASSAULT                                   4
Name: count, dtype: int64


In [77]:

print(df_cleaned['CrimeType'].value_counts())


CrimeType
OFFENSE_THEFT/OTHER                   13015
OFFENSE_THEFT F/AUTO                   6680
OFFENSE_MOTOR VEHICLE THEFT            5127
OFFENSE_ROBBERY                        2109
OFFENSE_ASSAULT W/DANGEROUS WEAPON     1026
OFFENSE_BURGLARY                       1004
OFFENSE_HOMICIDE                        187
OFFENSE_SEX ABUSE                       142
ASSAULT                                   4
Name: count, dtype: int64


In [94]:

categorical_cols = X.select_dtypes(include=['object']).columns

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)





In [96]:

print(X.isnull().sum())


X = X.fillna(X.mean())


WARD                                            0
ANC                                             0
DISTRICT                                        0
PSA                                             0
NEIGHBORHOOD_CLUSTER                            0
LATITUDE                                        0
LONGITUDE                                       0
YEAR                                            0
MONTH                                           0
DAY                                             0
HOUR                                            0
SHIFT_EVENING                                   0
SHIFT_MIDNIGHT                                  0
METHOD_KNIFE                                    0
METHOD_OTHERS                                   0
OFFENSE_BURGLARY                                0
OFFENSE_HOMICIDE                                0
OFFENSE_MOTOR VEHICLE THEFT                     0
OFFENSE_ROBBERY                                 0
OFFENSE_SEX ABUSE                               0


In [100]:
print(y_train.value_counts())


ASSAULT
1    23435
Name: count, dtype: int64


In [102]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


rf_model = RandomForestClassifier(random_state=42)


rf_model.fit(X_train, y_train)


rf_y_pred = rf_model.predict(X_test)


rf_accuracy = accuracy_score(y_test, rf_y_pred)


print(f"Random Forest Modeli Doğruluğu: {rf_accuracy}")


Random Forest Modeli Doğruluğu: 1.0


In [106]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


knn_model = KNeighborsClassifier(n_neighbors=5)


knn_model.fit(X_train, y_train)

knn_y_pred = knn_model.predict(X_test)


knn_accuracy = accuracy_score(y_test, knn_y_pred)
print(f"KNN Modeli Doğruluğu: {knn_accuracy}")



KNN Modeli Doğruluğu: 1.0


In [110]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


dt_model = DecisionTreeClassifier(random_state=42)


dt_model.fit(X_train, y_train)


dt_y_pred = dt_model.predict(X_test)


dt_accuracy = accuracy_score(y_test, dt_y_pred)
print(f"Decision Tree Modeli Doğruluğu: {dt_accuracy}")


Decision Tree Modeli Doğruluğu: 1.0


In [112]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score


nb_model = GaussianNB()


nb_model.fit(X_train, y_train)


nb_y_pred = nb_model.predict(X_test)


nb_accuracy = accuracy_score(y_test, nb_y_pred)
print(f"Naive Bayes Modeli Doğruluğu: {nb_accuracy}")


Naive Bayes Modeli Doğruluğu: 1.0


In [116]:
# Logistic Regression (Lojistik Regresyon) ve Support Vector Machine (SVM - Destek Vektör Makinesi) hesaplanamaz çünkü biz sadece assult suçuna göre bir sınıflandırma yaptık fakat Lojistik regresyon ve SVM - Destek Vektör Makinesi , en az iki sınıf (etiket) gerektirir, ancak tek bir sınıf varsa model eğitilemez.